# FEMI Transfer — ContractNLI, FEVEROUS and cross-model

Upload the execution ZIP first. Dataset files are NOT included. Every live inference cell below requires an explicit user action.

In [ ]:
from google.colab import files
import pathlib, zipfile, os, subprocess
uploaded=files.upload()
packet=next((x for x in uploaded if x.startswith("FEMI_NEXT_EXPERIMENTS_") and x.endswith(".zip")),None)
assert packet, "Upload FEMI_NEXT_EXPERIMENTS_20261108.zip"
with zipfile.ZipFile(packet) as z:z.extractall("/content")
root=pathlib.Path("/content/FEMI_NEXT_EXPERIMENTS_20261108")
os.chdir(root)
print(root)

## 1 — Offline suite (zero API calls)

In [ ]:
subprocess.run(["python","-m","unittest","discover","-s","tests","-v"],check=True)

## 2 — Upload the official ContractNLI dataset (no dataset bundled)
Open https://stanfordnlp.github.io/contract-nli/ and review the download terms, then download **contract-nli.zip** (or extract **test.json**).
The next cell accepts either file, automatically selects the annotated JSON split and sets `CNLI_JSON`.
You do **not** need FEVEROUS files at this stage. No paid API calls happen in this step.



In [ ]:
# Find the dataset if it is already present, otherwise ask you to upload it.
from google.colab import files as colab_files
from pathlib import Path
import subprocess

possible_sources = [
    root / 'datasets/contract-nli/test.json',
    root / 'contract-nli.zip',
    root / 'test.json',
    Path('/content/contract-nli.zip'),
    Path('/content/contract-nli/test.json'),
    Path('/content/test.json'),
    Path('/content/drive/MyDrive/contract-nli.zip'),
    Path('/content/drive/MyDrive/contract-nli/test.json'),
]
source = next((p for p in possible_sources if p.is_file()), None)
if source is None:
    print('Upload the OFFICIAL contract-nli.zip or extracted test.json (not the FEMI packet ZIP).')
    uploaded_contractnli = colab_files.upload()
    if not uploaded_contractnli:
        raise FileNotFoundError('No dataset uploaded. See https://stanfordnlp.github.io/contract-nli/')
    filenames = [Path(k) for k in uploaded_contractnli]
    candidates = [p for p in filenames if p.name in ('contract-nli.zip', 'test.json', 'dev.json')]
    if not candidates:
        candidates = [p for p in filenames if p.suffix.lower() in ('.zip', '.json')]
    if not candidates:
        raise ValueError('Upload the official ContractNLI ZIP or test.json/dev.json')
    source = candidates[0]

CNLI_JSON = str(root / 'datasets/contract-nli/test.json')
subprocess.run(['python', 'src/contractnli_inputs.py', '--input', str(source),
                '--output', CNLI_JSON], check=True)
print('Ready for the next builder cell: CNLI_JSON =', CNLI_JSON)


## 3 — Freeze the ContractNLI request bank (no API calls)
The previous cell must finish successfully before running this one. Inspect `runs/cnli/cases_PRIVATE.jsonl` before any paid inference.


In [ ]:
assert pathlib.Path(CNLI_JSON).is_file(), "Run dataset upload cell first"
subprocess.run(["python","src/build_transfer.py","contract","--input",CNLI_JSON,"--out","runs/cnli","--n","48"],check=True)
subprocess.run(["python","src/run_and_score.py","preflight","--dir","runs/cnli"],check=True)
print("ContractNLI manifest ready. Review the private case bank before any live calls.")


## 4 — OPTIONAL paid calls: explicit execute only
Do not run before completing the data/prompt audit and securing an OpenRouter key in Colab Secrets. Never paste the key into notebook source. The default command is dry-run.

In [ ]:
from google.colab import userdata
secret=userdata.get("OPENROUTER_API_KEY")
assert secret,"Set the secret in Colab Secrets"
os.environ["OPENROUTER_API_KEY"]=secret
del secret
subprocess.run(["python","src/run_and_score.py","run","--dir","runs/cnli","--models","models.anchor.json"],check=True)  # dry-run ONLY

In [ ]:
# PAID LIVE COLLECTION: deliberately commented; remove the leading # only after auditing the manifest.
# subprocess.run(["python","src/run_and_score.py","run","--dir","runs/cnli","--models","models.anchor.json","--execute"],check=True)

## 5 — After all 288 responses complete

In [ ]:
# subprocess.run(["python","src/run_and_score.py","analyze","--dir","runs/cnli","--models","models.anchor.json"],check=True)

## 6 — FEVEROUS and cross-model stages
After the ContractNLI analysis, repeat the process with FEVEROUS. For cross-model, build a frozen common subset and rerun the anchor contemporaneously with Llama and Gemini. See README for commands.

In [ ]:
# OFFLINE BUILD, after you have sourced and audited the input files:
# subprocess.run(["python","src/build_transfer.py","feverous","--input",FEVEROUS_DEV,"--wiki-jsonl",FEVEROUS_WIKI,"--out","runs/feverous","--n","48","--min-table","12"],check=True)
# subprocess.run(["python","src/build_cross_model.py","--contract-dir","runs/cnli","--feverous-dir","runs/feverous","--per-domain","24","--out","runs/cross"],check=True)